In [ ]:
import pandas as pd
import os
import json

In [ ]:
# --------------------- JSON STATS SETTINGS --------------------- #
DATASET = "cub"  # Options: "cxr" or "cub"
NET_CHOICE = "CNN" # Options: "Transformer", "Mamba", "CNN"
BASE_WEIGHTS_DIR = "./drive_folder/Bridging Human and Model Attention_ Explainability Analysis of CNN, Mamba, and ViT Architectures with Gaze-Based Validation"

data_folder_name = "CUB_200_2011" if DATASET == "cub" else "CXR"
file_path = os.path.join(BASE_WEIGHTS_DIR, NET_CHOICE, 'output_heatmaps', data_folder_name, 'heatmap_scores.json')
print("Loading JSON from:", file_path)


In [ ]:
# Load your data 
with open(file_path, 'r') as f:
    data = json.load(f)

rows = []

for filename, content in data.items():
    # 1. Extract the shared metadata
    index_val = content.get('index')
    train_val = content.get('train')
    
    # 2. Iterate through the keys to find the explainability methods
    # We ignore 'index' and 'train' keys
    for key, value in content.items():
        print(key)

        if key not in ['index', 'train']:
            model = key
            print(model)

            if model == "ResNet50":
                for cam_method in value.keys():
                    metrics = value.get(cam_method)
                    row = {
                        'name': filename,
                        'index': index_val,
                        'train': train_val,
                        'model': model,
                        'explainability_method': cam_method,
                        'JSS': metrics.get('JSS'),
                        'Chi2': metrics.get('Chi2'),
                        'PCC': metrics.get('PCC')
                    }
                    rows.append(row)
            else:
                print(value)
                metrics = value
                
                row = {
                    'name': filename,
                    'index': index_val,
                    'train': train_val,
                    'model': model,
                    'explainability_method': "x",
                    'JSS': metrics.get('JSS'),
                    'Chi2': metrics.get('Chi2'),
                    'PCC': metrics.get('PCC')
                }
                rows.append(row)
            

# 4. Create the DataFrame
df = pd.DataFrame(rows)

# Optional: Set the column order exactly as requested
df = df[['name', 'index', 'train', 'explainability_method', 'JSS', 'Chi2', 'PCC']]

print(df.head())

In [ ]:
#mean scores divided between explainability method and train/test set
cols_to_mean = ['JSS', 'Chi2', 'PCC']

mean_scores = df.groupby(['explainability_method', 'train'])[cols_to_mean].mean()
std_scores = df.groupby(['explainability_method', 'train'])[cols_to_mean].std()

print(mean_scores)
print(std_scores)

In [ ]:
import pandas as pd

In [ ]:
# --------------------- COMPARISON CSV STATS SETTINGS --------------------- #
FOLDER_1 = "CNN"
FOLDER_2 = "Transformer"
OUTPUT_TYPE = "heatmaps" # Options: "gaze" or "heatmaps"

csv_file_path = f"heatmap_comparison_results/{FOLDER_1}_{FOLDER_2}_{DATASET}_{OUTPUT_TYPE}.csv"
print("Loading CSV from:", csv_file_path)


In [ ]:
df = pd.read_csv(csv_file_path)
df.set_index('filename', inplace=True)

In [ ]:
mean = df.mean(numeric_only=True)
std = df.std(numeric_only=True)

print(mean)
print(std)